# 00 快速开始

按单元格依次运行，完成时间切分、分箱、评分卡、模型校准、规则挖掘与报告保存。仅使用基础模型依赖；首次使用先安装 `pip install -e ".[dev]"`。所有输出默认写入独立临时目录，不覆盖仓库已有报告。


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import os
import sys

repository_root = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "hscredit").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("请从仓库根目录或 examples 目录启动 Notebook")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
output_dir = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT") or mkdtemp(prefix="hscredit_quickstart_"))
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split as sklearn_train_test_split

from hscredit.core.binning import OptimalBinning
from hscredit.core.model_selection import time_train_test_split
from hscredit.core.models import RandomForest, ScoreCard
from hscredit.core.models.calibration import ProbabilityCalibrator
from hscredit.core.selectors import IVSelector, VIFSelector, CompositeFeatureSelector
from hscredit.report import auto_model_report
from hscredit.report.mining import SingleFeatureRuleMiner, TreeRuleExtractor


## 1. 构造演示数据并划分训练、校准、测试集

测试集只用于最后评估；校准集与模型训练集分开。


In [ ]:
def make_demo_data(n_samples: int = 400) -> pd.DataFrame:
    """构造不依赖外部文件的演示数据."""
    X, y = make_classification(
        n_samples=n_samples,
        n_features=4,
        n_informative=3,
        n_redundant=0,
        weights=[0.72, 0.28],
        random_state=42,
    )
    df = pd.DataFrame(X, columns=["年龄", "收入", "负债率", "申请次数"])
    df["申请日期"] = pd.date_range("2024-01-01", periods=n_samples, freq="D")
    df["客户编号"] = [f"C{i // 2:04d}" for i in range(n_samples)]
    df["目标"] = y
    return df
df = make_demo_data()
train_df, test_df = time_train_test_split(df, "申请日期", test_size=0.25)
features = ["年龄", "收入", "负债率", "申请次数"]
X_train, y_train = train_df[features], train_df["目标"]
X_test, y_test = test_df[features], test_df["目标"]
X_model_train, X_calib, y_model_train, y_calib = sklearn_train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)


## 2. 分箱、特征筛选与 WOE 评分卡

原始数据和 WOE 数据的输入类型需要明确，不能用分箱索引替代 WOE。


In [ ]:
binner = OptimalBinning(method="best_iv", max_n_bins=5, n_jobs=1)
binner.fit(X_train, y_train)
X_train_woe = binner.transform(X_train, metric="woe")

selector = CompositeFeatureSelector(
    [
        ("iv", IVSelector(threshold=0.0)),
        ("vif", VIFSelector(threshold=20.0)),
    ]
)
X_selected = selector.fit_transform(X_train_woe, y_train)

scorecard = ScoreCard(binner=binner)
scorecard.fit(X_train_woe, y_train)
scores = scorecard.predict(X_test)


## 3. 分类模型与独立概率校准


In [ ]:
model = RandomForest(n_estimators=30, random_state=42, n_jobs=1)
model.fit(X_model_train, y_model_train)
metrics = model.evaluate(X_test, y_test)

calibrator = ProbabilityCalibrator(
    model=model,
    method="platt",
    calib_ratio=None,
).fit(X_calib, y_calib)
calibration_report = calibrator.report(X_test, y_test)


## 4. 规则挖掘


In [ ]:
rule_df = train_df[features + ["目标"]]
single_miner = SingleFeatureRuleMiner(
    target="目标",
    method="best_iv",
    max_n_bins=4,
    min_samples=5,
).fit(rule_df)
single_rules = single_miner.get_top_rules(top_n=5)

tree_extractor = TreeRuleExtractor(max_depth=3, min_samples_leaf=10)
tree_extractor.fit(X_train, y_train)
tree_rules = tree_extractor.extract_rules()


## 5. 保存模型和报告


In [ ]:
artifact_path = None
report_path = None
if output_dir is not None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    artifact_path = model.save_artifact(output_dir / "risk_model.joblib")
    report_path = output_dir / "quickstart_model_report.xlsx"

report = auto_model_report(
    model,
    X_train=X_model_train,
    y_train=y_model_train,
    X_test=X_test,
    y_test=y_test,
    excel_path=report_path,
    verbose=False,
    with_plots=False,
)
report_metrics = report.get_metrics()
report_summary = report.summary()

if report_path is not None:
    assert report_path.is_file() and report_path.stat().st_size > 0, "模型报告未正确生成"

result = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "calibration_rows": len(X_calib),
    "selected_features": list(X_selected.columns),
    "scores": scores,
    "metrics": metrics,
    "calibration_report": calibration_report,
    "single_rules": single_rules,
    "tree_rules": tree_rules,
    "report": report,
    "report_path": report_path,
    "report_metrics": report_metrics,
    "report_summary": report_summary,
    "artifact_path": artifact_path,
}


## 6. 验证结果

更完整的字段、正类概率、调参保留策略、轻量推理制品见 `28_model_workflow.ipynb`。


In [ ]:
assert result["train_rows"] == 300
assert result["test_rows"] == 100
assert result["calibration_rows"] == 60
assert np.isfinite(result["scores"]).all()
assert Path(result["artifact_path"]).is_file()
print("快速开始 Notebook 执行成功")
display(result["report_metrics"])
display(result["calibration_report"])
